In [1]:
# Install uv globally inside the Colab instance
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.1/27.1 MB 81.5 MB/s eta 0:00:00


In [2]:
# Force uv to install packages directly into Colab's system environment
!uv pip install --system "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl peft transformers bitsandbytes datasets accelerate xformers

Using Python 3.12.13 environment at: /usr
Resolved 103 packages in 11.84s
Prepared 13 packages in 8.77s
Uninstalled 4 packages in 575ms
Installed 13 packages in 106ms
 + bitsandbytes==0.49.2
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.3.0
 + hf-transfer==0.1.9
 + msgspec==0.21.1
 - pyarrow==18.1.0
 + pyarrow==25.0.0
 - torchao==0.10.0
 + torchao==0.17.0
 - transformers==5.12.1
 + transformers==5.5.0
 + trl==0.24.0
 + tyro==1.0.15
 + unsloth==2026.7.2 (from git+https://github.com/unslothai/unsloth.git@6412efd7d9ae98ea736110979cf6042b691a2249)
 + unsloth-zoo==2026.7.2
 + xformers==0.0.35


In [3]:
import os
import shutil
from google.colab import files

# 1. Create the target data directory in Colab
os.makedirs("data", exist_ok=True)
print("📂 Created './data/' directory.")

# 2. Trigger the local file upload prompt
print("👇 Click 'Choose Files' below and select your 3 data files together:")
uploaded = files.upload()

# 3. Expected filenames based on the assignment layout
expected_files = [
    "non_instruction_data.txt",
    "instruction_dataset.jsonl",
    "preference_dataset.jsonl"
]

# 4. Route files to the correct folder automatically
print("\n🔄 Organizing files...")
for file_name in uploaded.keys():
    if file_name in expected_files:
        dest_path = os.path.join("data", file_name)
        shutil.move(file_name, dest_path)
        print(f"✅ Successfully moved '{file_name}' ➡️ '{dest_path}'")
    else:
        print(f"⚠️ Warning: '{file_name}' was uploaded but doesn't match the required file names. Skipped.")

print("\n🎉 Setup complete! Your data tier is ready for training.")

📂 Created './data/' directory.
👇 Click 'Choose Files' below and select your 3 data files together:


Saving instruction_dataset.jsonl to instruction_dataset.jsonl
Saving non_instruction_data.txt to non_instruction_data.txt
Saving preference_dataset.jsonl to preference_dataset.jsonl

🔄 Organizing files...
✅ Successfully moved 'instruction_dataset.jsonl' ➡️ 'data/instruction_dataset.jsonl'
✅ Successfully moved 'non_instruction_data.txt' ➡️ 'data/non_instruction_data.txt'
✅ Successfully moved 'preference_dataset.jsonl' ➡️ 'data/preference_dataset.jsonl'

🎉 Setup complete! Your data tier is ready for training.


In [4]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048 # Supports arbitrary scaling internal to Unsloth architectures
dtype = None          # Auto-detects hardware constraints (Float16 for T4, Bfloat16 for Ampere+)
load_in_4bit = True   # Utilizes 4-bit QLoRA normalization to minimize VRAM allocations

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B", # Highly optimized domain-base framework
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.aria.image_processing_pil_aria`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.auto.image_processing_auto`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_beit`. Returning `is_flash_linear_attention_available` instead. Behavior may be different and this alias will be removed in future versions.
Accessing `is_flash_linear_attention_available` from `.models.beit.image_processing_pil_beit`. R

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [5]:
from datasets import load_dataset

# Ingest raw domain text directly from the Colab local data structure
data_path = "./data/non_instruction_data.txt"

# Load line-separated text files into standard Hugging Face dataset format
dataset = load_dataset("text", data_files={"train": data_path})

# Clean text by removing trailing whitespaces and isolating structural text tokens
def clean_and_chunk(examples):
    cleaned_texts = [text.strip() for text in examples["text"] if len(text.strip()) > 10]
    return {"text": cleaned_texts}

processed_dataset = dataset["train"].map(clean_and_chunk, batched=True, remove_columns=["text"])
print(f"🎉 Successfully loaded and structured {len(processed_dataset)} raw domain paragraphs.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

🎉 Successfully loaded and structured 50 raw domain paragraphs.


In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank matrix depth factor
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32, # Scaling matrix variable coefficient
    lora_dropout = 0, # Optimized to 0 for maximum speed within Unsloth pipelines
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Massive VRAM preservation patch
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.7.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [7]:
from trl import SFTConfig, SFTTrainer
import torch

# Use modern SFTConfig instead of standard TrainingArguments to stabilize configurations
training_args = SFTConfig(
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    max_steps = 40,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    output_dir = "outputs_raw_tuning",
    dataset_text_field = "text",   # Modern TRL moves these directly inside the config block
    max_seq_length = max_seq_length,
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = processed_dataset,
    packing = False,
    args = training_args,
    # dataset_num_proc is removed here to clear out the multi-process PicklingError
)

# Start Stage 1 Training Loop
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/50 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 50 | Num Epochs = 10 | Total steps = 40
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.102256
2,3.555228
3,3.470120
4,1.979697
5,3.149359
6,2.708964
7,2.891743
8,2.664266
9,2.423568
10,2.277097


Unsloth: Restored added_tokens_decoder metadata in outputs_raw_tuning/checkpoint-40/tokenizer_config.json.


In [8]:
# Save local PEFT adapter adjustments and target configuration files
model.save_pretrained_merged("./models/stage1_raw", tokenizer, save_method = "lora")
print("💾 Stage 1 non-instruction adapter successfully compiled and saved to standard directory.")

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ./models/stage1_raw/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:44<00:00, 44.95s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:22<00:00, 22.17s/it]


Unsloth: Merge process complete. Saved to `/content/models/stage1_raw`
💾 Stage 1 non-instruction adapter successfully compiled and saved to standard directory.


In [9]:
# Re-activate optimized internal inference loops
FastLanguageModel.for_inference(model)

# Supply a raw concept query to observe predictive sentence completion
test_prompt = "Dynamic CVV systems enhance card security by"
inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 60, use_cache = True)
generated_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("📝 Test Text Input Target:")
print(test_prompt)
print("\n🔮 Model Sequence Completion Output:")
print(generated_output)

Both `max_new_tokens` (=60) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


📝 Test Text Input Target:
Dynamic CVV systems enhance card security by

🔮 Model Sequence Completion Output:
Dynamic CVV systems enhance card security by continuously changing the security code via digital displays on the card or within a banking app. This dynamic feature requires in-store terminals to also update, ensuring all transactions validate the latest three digits. If a cardholder mistypes the new CVV, transactions may fail, necessitating contactless or in-person card


In [10]:
import os
import torch
from unsloth import FastLanguageModel

# Ensure the reports directory exists inside your Colab instance
os.makedirs("reports", exist_ok=True)

# 1. Define the 10 assignment validation questions
eval_questions = [
    "What is the primary difference between a credit card and a debit card?",
    "How can I avoid paying interest on my credit card balance?",
    "What should I do if my debit card is lost or stolen?",
    "What is an APR on a credit card?",
    "Will using a debit card help me build my credit score?",
    "What happens if I only pay the minimum amount due on my credit card?",
    "How do I dispute an unauthorized charge on my credit card?",
    "Can I use my debit card for international transactions?",
    "What is a credit card grace period?",
    "What is an overdraft fee on a debit card?"
]

print("📥 Loading original un-tuned Base Model from Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

print("🔮 Generating baseline responses...")
markdown_output = "# Base Model Baseline Evaluation Report\n\n"
markdown_output += "| Question | Base Model Answer | Problem |\n"
markdown_output += "| :--- | :--- | :--- |\n"

for question in eval_questions:
    inputs = tokenizer([question], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=120, use_cache=True)
    raw_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # Clean string format for clear markdown table row structure
    clean_answer = raw_output.replace(question, "").strip().replace("\n", " ")
    markdown_output += f"| {question} | {clean_answer} | **Generic Response:** Lacks domain-specific alignment. |\n"

# Save directly to the required assignment directory track
with open("reports/base_model_evaluation.md", "w", encoding="utf-8") as f:
    f.write(markdown_output)

print("💾 File successfully saved to 'reports/base_model_evaluation.md'!")

📥 Loading original un-tuned Base Model from Unsloth...
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🔮 Generating baseline responses...


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

💾 File successfully saved to 'reports/base_model_evaluation.md'!
